In [ ]:
!pip install transformers accelerate pillow opencv-python

In [ ]:
import requests
import base64
import cv2
import numpy as np
from PIL import Image


# -----------------------------
# Image Preprocessing
# -----------------------------
def preprocess_image(image_path):
    img = cv2.imread(image_path)

    # convert to grayscale
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # reduce noise
    blur = cv2.GaussianBlur(gray, (5, 5), 0)

    # improve contrast
    thresh = cv2.adaptiveThreshold(
        blur,
        255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY,
        11,
        2
    )

    # resize to improve readability
    resized = cv2.resize(thresh, None, fx=1.5, fy=1.5)

    return resized


# -----------------------------
# Encode Image
# -----------------------------
def encode_image_from_array(img_array):
    _, buffer = cv2.imencode('.jpg', img_array)
    return base64.b64encode(buffer).decode()


# -----------------------------
# LLM Extraction (Ollama)
# -----------------------------
def extract_drugs_with_ollama(image_base64):
    prompt = """
You are a medical assistant.

Extract ONLY drug names from this prescription image.

Strict rules:
- Do NOT guess
- Do NOT add drugs not visible
- Ignore unclear words
- Ignore numbers and dosage
- Return ONLY a python list

Example:
["Paracetamol", "Augmentin"]
"""

    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": "deepseek-ocr",
            "prompt": prompt,
            "images": [image_base64],
            "stream": False
        }
    )

    return response.json()["response"]

# -----------------------------
# Main Pipeline
# -----------------------------
def extract_drugs(image_path):
    # preprocess
    processed_img = preprocess_image(image_path)

    # encode
    base64_img = encode_image_from_array(processed_img)

    # extract
    result = extract_drugs_with_ollama(base64_img)

    return result

# -----------------------------
# Run Example
# -----------------------------
if __name__ == "__main__":
    image_path = "image.png"
    result = extract_drugs(image_path)

    print("Extracted Drugs:")
    print(result)